In [ ]:
import polars as pl
import sqlalchemy as sa
from pydantic import BaseModel
from IPython.display import display
import uuid
import random
from typing import Optional


DB_HOST = '100.95.220.1'
DB_PORT = 5432
DB_USER = 'admin'
DB_PASSWORD = "password"
DB_CONECTION = "athena"

connection_url = (
    f'postgresql://{DB_USER}:{DB_PASSWORD}'
    f'@{DB_HOST}:{DB_PORT}/{DB_CONECTION}'
)
engine = sa.create_engine(connection_url)

class Carrera(BaseModel):
    id_carrera: int
    nombre_carrera: str

class Habilidad(BaseModel):
    id_habilidad: int
    descripcion: str

class Usuario(BaseModel):
    id_usuario: Optional[int] = None
    uuid_usuario: Optional[str] = None
    nombre: str
    correo: str

class Resultado(BaseModel):
    id_resultado: Optional[int] = None
    uuid_usuario: str
    id_carrera: int
    porcentaje_coincidencia: float = 100.0
    top: int = 1

def get_all_careers() -> list[Carrera]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text("""
            SELECT id_carrera, nombre_carrera
            FROM carrera
            ORDER BY id_carrera;
        """)).mappings().all()
    return [
        Carrera(id_carrera=row["id_carrera"], nombre_carrera=row["nombre_carrera"])
        for row in rows
    ]

def get_habilities_by_career(id_carrera: int) -> list[Habilidad]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text("""
            SELECT h.id_habilidad, h.descripcion
            FROM carrera_habilidad ch
            JOIN habilidad h ON h.id_habilidad = ch.id_habilidad
            WHERE ch.id_carrera = :id_carrera
            ORDER BY h.id_habilidad;
        """), {"id_carrera": id_carrera}).mappings().all()
    return [
        Habilidad(id_habilidad=row["id_habilidad"], descripcion=row["descripcion"])
        for row in rows
    ]

def generate_name_and_email() -> tuple[str, str]:
    nombres = ["Ana", "Carlos", "María", "Juan", "Sofía"]
    apellidos = ["Gómez", "Pérez", "López", "Rodríguez", "Martínez"]
    nombre_aleatorio = random.choice(nombres)
    apellido_aleatorio = random.choice(apellidos)
    name = f"{nombre_aleatorio} {apellido_aleatorio}"
    email = generate_email(nombre_aleatorio, apellido_aleatorio)
    return name, email

def generate_email(name: str, apellido: str) -> str:
    return f"{name.lower().replace(' ', '.')}.{apellido.lower()}@athena.edu"

def submit_user(user: Usuario) -> Usuario:
    with engine.begin() as conn:
        existing_user = conn.execute(sa.text("""
            SELECT id_usuario, uuid_usuario
            FROM usuario
            WHERE correo = :correo;
        """), {"correo": user.correo}).mappings().first()

        if existing_user is not None:
            conn.execute(sa.text("""
                UPDATE usuario
                SET nombre = :nombre
                WHERE id_usuario = :id_usuario;
            """), {"nombre": user.nombre, "id_usuario": existing_user["id_usuario"]})
            return Usuario(
                id_usuario=existing_user["id_usuario"],
                uuid_usuario=str(existing_user["uuid_usuario"]),
                nombre=user.nombre,
                correo=user.correo,
            )

        row = conn.execute(sa.text("""
            INSERT INTO usuario (nombre, correo)
            VALUES (:nombre, :correo)
            RETURNING id_usuario, uuid_usuario;
        """), {"nombre": user.nombre, "correo": user.correo}).mappings().one()
        return Usuario(
            id_usuario=row["id_usuario"],
            uuid_usuario=str(row["uuid_usuario"]),
            nombre=user.nombre,
            correo=user.correo,
        )

def submit_user_habilities(id_usuario: int, habilidad_ids: list[int]) -> None:
    with engine.begin() as conn:
        conn.execute(sa.text("""
            DELETE FROM usuario_habilidad
            WHERE id_usuario = :id_usuario;
        """), {"id_usuario": id_usuario})

        if not habilidad_ids:
            return

        conn.execute(sa.text("""
            INSERT INTO usuario_habilidad
                (id_usuario, id_habilidad, cumplimiento_criterio)
            VALUES (:id_usuario, :id_habilidad, 100.0);
        """), [
            {"id_usuario": id_usuario, "id_habilidad": id_habilidad}
            for id_habilidad in habilidad_ids
        ])

def construct_result(uuid_usuario: str, id_carrera: int, porcentaje_coincidencia: float = 100.0, top: int = 1) -> Resultado:
    with engine.begin() as conn:
        conn.execute(sa.text("""
            INSERT INTO resultado
                (uuid_usuario, id_carrera, porcentaje_coincidencia, top)
            VALUES (:uuid_usuario, :id_carrera, :porcentaje_coincidencia, :top)
            ON CONFLICT (uuid_usuario, id_carrera) DO UPDATE SET
                porcentaje_coincidencia = EXCLUDED.porcentaje_coincidencia,
                top = EXCLUDED.top;
        """), {
            "uuid_usuario": uuid_usuario,
            "id_carrera": id_carrera,
            "porcentaje_coincidencia": porcentaje_coincidencia,
            "top": top,
        })
    return Resultado(
        uuid_usuario=uuid_usuario,
        id_carrera=id_carrera,
        porcentaje_coincidencia=porcentaje_coincidencia,
        top=top,
    )

def run_happy_path() -> pl.DataFrame:
    carreras = get_all_careers()

    for carrera in carreras:
        nombre, correo = generate_name_and_email()
        user = Usuario(nombre=nombre, correo=correo)
        user_db = submit_user(user)
        habilidades = get_habilities_by_career(carrera.id_carrera)
        habilidad_ids = [habilidad.id_habilidad for habilidad in habilidades]
        submit_user_habilities(user_db.id_usuario, habilidad_ids)
        construct_result(user_db.uuid_usuario, carrera.id_carrera)

    query = """
        SELECT
            u.id_usuario,
            u.nombre AS nombre_usuario,
            u.uuid_usuario,
            c.id_carrera,
            c.nombre_carrera AS carrera,
            r.porcentaje_coincidencia,
            h.id_habilidad,
            h.descripcion AS habilidad,
            uh.cumplimiento_criterio
        FROM resultado r
        JOIN usuario u ON u.uuid_usuario = r.uuid_usuario
        JOIN carrera c ON c.id_carrera = r.id_carrera
        JOIN carrera_habilidad ch ON ch.id_carrera = c.id_carrera
        JOIN habilidad h ON h.id_habilidad = ch.id_habilidad
        JOIN usuario_habilidad uh
            ON uh.id_usuario = u.id_usuario
            AND uh.id_habilidad = h.id_habilidad
        ORDER BY c.id_carrera, u.id_usuario, h.id_habilidad;
    """

    with engine.begin() as conn:
        return pl.read_database(query, connection=conn)

if __name__ == "__main__":
    resultados_df = run_happy_path()
    print(f"Carreras procesadas: {len(get_all_careers())}")
    print(f"Filas de habilidades/resultados: {resultados_df.height}")
    display(resultados_df)

ModuleNotFoundError: No module named 'psycopg2'